In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    try:
        %reload_ext rpy2.ipython
    except Exception as e2:
        print("Note on rpy2 initialization:", e2)

# ESP32-Optimized Centroid-Based Local Outlier Factor (LOF) for ESI 1 Detection (`models/lof_esi1_anomaly_detection.ipynb`)

This notebook implements a **Microcontroller-Compatible Centroid-Based Local Outlier Factor (LOF) Anomaly Detector** to address the extreme class imbalance of ESI 1 (~1% of ED admissions vs 99% non-ESI 1):

### ESP32 Microcontroller Compression Strategy
1. **The ESP32 Memory Constraint**:
   - Standard LOF requires storing all 546,862 training samples ($546,862 \times 35 \text{ floats} \approx 76.5 \text{ MB}$ RAM), which exceeds ESP32's 512 KB SRAM capacity.
2. **Centroid Exemplar Reduction ($K = 200$)**:
   - Extracts $K = 200$ reference centroids from non-ESI 1 training data using K-Means clustering.
   - **Flash Memory Footprint**: $200 \times 35 \text{ double floats} = 7,000 \text{ numbers} \approx \mathbf{28 \text{ KB}}$ Flash array!
   - Fits easily inside ESP32 Flash memory while retaining the density structure of normal triage vitals.
3. **Runtime Anomaly Scoring**:
   - For an incoming patient vital vector $x$, the C engine computes distances to all 200 centroids, extracts the $k$-nearest neighbors ($k = 10$), computes Local Reachability Density (LRD), and outputs an LOF anomaly score.
   - An LOF score $> \text{threshold}$ flags the patient as an **ESI 1 Critical Anomaly**.

### Pipeline Steps
1. **Load Data & 35 Features in R**: Reads dataset and constructs 35 predictor features.
2. **Python Centroid Extraction & LOF Model**: Fits K-Means ($K=200$) and calculates baseline LRDs.
3. **Transpile Centroids & LOF Engine to C**: Writes static 28 KB C header `deploy/esi1_lof_detector.c` and compiles `deploy/esi1_lof_detector.so`.
4. **C Shared Library Inference & Audit**: Runs C inference via `ctypes` on Holdout Test Set.
5. **Benchmark Metrics**: Evaluates **Recall**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC** for ESI 1 detection, saving report to `reports/esi1_lof_test_report.csv`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Prepare 35 Predictor Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(pROC)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age = raw_df$age, gender = gender_vec, cc_breathingdifficulty = cc_bd_vec,
  triage_vital_hr = t_hr, triage_vital_sbp = t_sbp, triage_vital_rr = t_rr, triage_vital_o2 = t_o2,
  pulse_last = p_last, resp_last = r_last, spo2_last = o2_last, sbp_last = s_last,
  pulse_min = p_min, resp_min = r_min, spo2_min = o2_min, sbp_min = s_min,
  pulse_max = p_max, resp_max = r_max, spo2_max = o2_max, sbp_max = s_max,
  hr_mean_to_last = t_hr - p_last, sbp_mean_to_last = t_sbp - s_last, spo2_mean_to_last = t_o2 - o2_last, rr_mean_to_last = t_rr - r_last,
  hr_range = p_max - p_min, rr_range = r_max - r_min, spo2_range = o2_max - o2_min, sbp_range = s_max - s_min,
  hr_last_to_min = p_last - p_min, rr_last_to_min = r_last - r_min, spo2_last_to_min = o2_last - o2_min, sbp_last_to_min = s_last - s_min,
  hr_last_to_max = p_last - p_max, rr_last_to_max = r_last - r_max, spo2_last_to_max = o2_last - o2_max, sbp_last_to_max = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Export partitions to R global environment
train_py <<- train_df
val_py   <<- val_df
test_py  <<- test_df
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Extract K=200 Reference Centroids from Non-ESI 1 Data & Calculate LRDs
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
# Retrieve dataframes
try:
    pandas2ri.activate()
    train_df = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['train_py']))
    val_df   = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['val_py']))
    test_df  = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['test_py']))
except Exception:
    train_df = pd.DataFrame(r['train_py'])
    val_df   = pd.DataFrame(r['val_py'])
    test_df  = pd.DataFrame(r['test_py'])
feature_cols = [c for c in train_df.columns if c != 'target_col']
binary_cols  = ['gender', 'cc_breathingdifficulty']
cont_cols    = [c for c in feature_cols if c not in binary_cols]
# Standard scale continuous features
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(train_df[cont_cols])
X_val_cont   = scaler.transform(val_df[cont_cols])
X_test_cont  = scaler.transform(test_df[cont_cols])
X_train = np.hstack([X_train_cont, train_df[binary_cols].values])
X_val   = np.hstack([X_val_cont,   val_df[binary_cols].values])
X_test  = np.hstack([X_test_cont,  test_df[binary_cols].values])
y_tr = train_df['target_col'].astype(str).values
y_ts = test_df['target_col'].astype(str).values
# Filter non-ESI 1 training data (normal triage baseline)
non_esi1_mask = (y_tr != '1')
X_non_esi1 = X_train[non_esi1_mask]
print(f"Training Non-ESI 1 dataset size: {X_non_esi1.shape[0]} samples")
# Extract K=200 Centroid Exemplars
K_CENTROIDS = 200
k_nn = 10
print(f"Extracting {K_CENTROIDS} Centroids using MiniBatchKMeans...")
kmeans = MiniBatchKMeans(n_clusters=K_CENTROIDS, batch_size=2048, random_state=42, n_init=3)
centroids = kmeans.fit(X_non_esi1).cluster_centers_
# Fit Nearest Neighbors on Centroids
nn = NearestNeighbors(n_neighbors=k_nn + 1, metric='euclidean')
nn.fit(centroids)
distances, indices = nn.kneighbors(centroids)
# distances to k-th neighbor (k-distance)
k_distances = distances[:, -1]
# Calculate reachability distance and LRD for centroids
centroid_lrd = np.zeros(K_CENTROIDS)
for i in range(K_CENTROIDS):
    reach_dists = []
    for idx_j in indices[i, 1:]:
        d_ij = np.linalg.norm(centroids[i] - centroids[idx_j])
        reach_d = max(k_distances[idx_j], d_ij)
        reach_dists.append(reach_d)
    avg_reach_d = np.mean(reach_dists)
    centroid_lrd[i] = 1.0 / (avg_reach_d + 1e-10)
print(f"Centroid LRD calculation complete! Flash footprint: {centroids.nbytes / 1024:.2f} KB")

In [ ]:
# ---------------------------------------------------------
# Step 3: Export Compact C Library for ESP32 (deploy/esi1_lof_detector.c)
# ---------------------------------------------------------
deploy_dir = "../deploy"
if not os.path.exists(deploy_dir):
    deploy_dir = "deploy"
os.makedirs(deploy_dir, exist_ok=True)
c_file_path  = os.path.abspath(os.path.join(deploy_dir, "esi1_lof_detector.c"))
so_file_path = os.path.abspath(os.path.join(deploy_dir, "esi1_lof_detector.so"))
# Format centroids and LRDs as static C arrays
centroids_c_str = ",\n    ".join([
    "{" + ", ".join([f"{val:.8f}" for val in row]) + "}"
    for row in centroids
])
lrd_c_str = ", ".join([f"{val:.8f}" for val in centroid_lrd])
c_code = f"""
/* =========================================================================
   ESP32 Microcontroller Centroid LOF ESI 1 Anomaly Detector
   ========================================================================= */
#include <math.h>
#include <stddef.h>
#include <stdlib.h>
#define NUM_CENTROIDS {K_CENTROIDS}
#define NUM_FEATS 35
#define K_NN 10
// 28 KB Flash Storage Centroid Matrix
static const double CENTROIDS[NUM_CENTROIDS][NUM_FEATS] = {{
    {centroids_c_str}
}};
static const double CENTROID_LRD[NUM_CENTROIDS] = {{
    {lrd_c_str}
}};
// Pair struct for k-NN sorting
typedef struct {{
    double dist;
    int idx;
}} DistIdxPair;
static int compare_pairs(const void * a, const void * b) {{
    double diff = ((DistIdxPair*)a)->dist - ((DistIdxPair*)b)->dist;
    return (diff > 0) - (diff < 0);
}}
double predict_esi1_lof_score(double * input, int * is_esi1_anomaly) {{
    DistIdxPair pairs[NUM_CENTROIDS];
    
    // 1. Compute Euclidean Distance from input sample to all 200 centroids
    for (int i = 0; i < NUM_CENTROIDS; i++) {{
        double d_sq = 0.0;
        for (int j = 0; j < NUM_FEATS; j++) {{
            double diff = input[j] - CENTROIDS[i][j];
            d_sq += diff * diff;
        }}
        pairs[i].dist = sqrt(d_sq);
        pairs[i].idx  = i;
    }}
    
    // 2. Sort distances to extract K_NN nearest centroids
    qsort(pairs, NUM_CENTROIDS, sizeof(DistIdxPair), compare_pairs);
    
    // 3. Compute Local Reachability Density (LRD) of sample
    double sum_reach_dist = 0.0;
    for (int i = 0; i < K_NN; i++) {{
        double k_dist_neighbor = 0.1; // Baseline centroid scale
        double reach_d = pairs[i].dist > k_dist_neighbor ? pairs[i].dist : k_dist_neighbor;
        sum_reach_dist += reach_d;
    }}
    double avg_reach_dist = sum_reach_dist / (double)K_NN;
    double sample_lrd = 1.0 / (avg_reach_dist + 1e-10);
    
    // 4. Compute LOF Anomaly Score
    double sum_lrd_ratio = 0.0;
    for (int i = 0; i < K_NN; i++) {{
        int c_idx = pairs[i].idx;
        sum_lrd_ratio += (CENTROID_LRD[c_idx] / (sample_lrd + 1e-10));
    }}
    double lof_score = sum_lrd_ratio / (double)K_NN;
    
    // 5. Anomaly Thresholding (LOF > 1.5 indicates ESI 1 extreme outlier)
    *is_esi1_anomaly = (lof_score > 1.5) ? 1 : 0;
    return lof_score;
}}
"""
with open(c_file_path, "w") as f:
    f.write(c_code)
print(f"ESI 1 Centroid LOF C Code written to: {c_file_path} ({len(c_code)} bytes)")
# Compile C Shared Library
import subprocess
compile_cmd = f"gcc -O3 -shared -fPIC -lm '{c_file_path}' -o '{so_file_path}'"
print(f"Executing: {compile_cmd}")
res = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if res.returncode == 0:
    print(f"SUCCESS: Dynamic C Shared Library compiled -> {so_file_path}")
else:
    raise RuntimeError(f"GCC Compilation Failed:\n{res.stderr}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Run C Shared Library Inference on Holdout Test Set & Evaluate Metrics
# ---------------------------------------------------------
import ctypes
from sklearn.metrics import confusion_matrix, roc_auc_score
c_lib = ctypes.CDLL(so_file_path)
c_lib.predict_esi1_lof_score.argtypes = [
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_int)
]
c_lib.predict_esi1_lof_score.restype = ctypes.c_double
N = X_test.shape[0]
lof_scores_c = np.zeros(N, dtype=np.float64)
esi1_preds_c = np.zeros(N, dtype=np.int32)
for i in range(N):
    x_sample = X_test[i, :].astype(np.float64)
    x_ptr = x_sample.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    out_flag = ctypes.c_int()
    
    score = c_lib.predict_esi1_lof_score(x_ptr, ctypes.byref(out_flag))
    lof_scores_c[i] = score
    esi1_preds_c[i] = out_flag.value
# Ground Truth for ESI 1
y_true_esi1 = (y_ts == '1').astype(int)
# Evaluate Anomaly Detection Metrics for ESI 1
cm = confusion_matrix(y_true_esi1, esi1_preds_c)
tn, fp, fn, tp = cm.ravel()
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
spec      = tn / (tn + fp) if (tn + fp) > 0 else 0.0
bal_acc   = (recall + spec) / 2.0
try:
    roc_auc = roc_auc_score(y_true_esi1, lof_scores_c)
except Exception:
    roc_auc = np.nan
print("============================================================")
print("   ESP32 CENTROID LOF ESI 1 ANOMALY DETECTOR BENCHMARK")
print("============================================================")
print(f"  ESI 1 Sensitivity (Recall)  : {recall:.4f}")
print(f"  Non-ESI 1 Specificity       : {spec:.4f}")
print(f"  Balanced Accuracy           : {bal_acc:.4f}")
print(f"  ESI 1 Detection ROC-AUC     : {roc_auc:.4f}")
print("============================================================\n")
print("Confusion Matrix (ESI 1 Anomaly Detection):")
print(cm)
# Save Report to reports/esi1_lof_test_report.csv
reports_dir = "../reports"
if not os.path.exists(reports_dir):
    reports_dir = "reports"
os.makedirs(reports_dir, exist_ok=True)
rep_df = pd.DataFrame({
    'Model': ['Centroid_LOF_ESP32'],
    'Recall_ESI1': [round(recall, 4)],
    'Specificity': [round(spec, 4)],
    'Balanced_Accuracy': [round(bal_acc, 4)],
    'ROC_AUC': [round(roc_auc, 4)]
})
rep_df.to_csv(os.path.join(reports_dir, "esi1_lof_test_report.csv"), index=False)
print(f"\nESI 1 LOF Test Report written to: {os.path.join(reports_dir, 'esi1_lof_test_report.csv')}")